In [1]:
import pandas as pd
import numpy as np
import os
import ansys.aedt.core
import math
import shutil
import time

In [2]:
## Desktop 만들고, Project와 Design 만들기 ##

# creating Desktop object
# write the aedt(ansys electronics desktop) version and whether you are using a student license
DT = ansys.aedt.core.Desktop(version="2025.2", non_graphical=False, student_version=True)

# Option to automatically save at regular intervals
# release it because The simulation is running on a script basis 
DT.disable_autosave()

# solution type
sol_type = "Magnetostatic"

# creating maxwell3D design object
# write solution type and whether you are using a student license
# 처음 design 객체를 생성하면 새로운 project와 함께 생성되고, project와 design은 임의의 이름을 가짐
M3D = ansys.aedt.core.maxwell.Maxwell3d(solution_type=sol_type, student_version=True)
# ansys 레코딩 코드를 사용하기 위한 odesign 객체
oDesign = M3D.odesign

PyAEDT INFO: Python version 3.12.13 | packaged by Anaconda, Inc. | (main, Jul  9 2026, 14:26:47) [MSC v.1942 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.15.3.
PyAEDT INFO: Initializing new Desktop session.
PyAEDT INFO: Log on console is enabled.
PyAEDT INFO: Log on file C:\Users\abane\AppData\Local\Temp\pyaedt_abane_4af602ca-07a5-45f7-9353-10559f8317ef.log is enabled.
PyAEDT INFO: Log on AEDT is disabled.
PyAEDT INFO: Debug logger is disabled. PyAEDT methods will not be logged.
PyAEDT INFO: Launching PyAEDT with gRPC plugin.
PyAEDT INFO: New AEDT session is starting on gRPC port 52410.
PyAEDT INFO: Electronics Desktop started on gRPC port: 52410 after 15.531598567962646 seconds.
PyAEDT INFO: AEDT installation Path C:\Program Files\ANSYS Inc\ANSYS Student\v252\AnsysEM
PyAEDT INFO: Ansoft.ElectronicsDesktop.2025.2 Student version started with process ID 21404.
PyAEDT INFO: Python version 3.12.13 | packaged by Anaconda, Inc. | (main, Jul  9 2026, 14:26:47) [MSC v.1942 64 bit (AMD64)].


In [4]:
## Project 폴더 만들기 ##

# project name
# change the name to what you want
proj_name = "ex1"

# path of project directory
dir = os.getcwd() + f"\\{proj_name}"
print(dir)

# # project 폴더 생성
# # 실행 전 이전 폴더 삭제 필요
# os.mkdir(dir)

# the name of maxwell3D design object
desi_name = "ex1"


c:\Users\abane\ansys\Doyoon\에너지 변환 공학\1주차\Ansys입문\05_Pyaedt_example(숙제)\ex1_ex2_Maxwell3D_Magnetostatic\M3D_ex1_Excitation\ex1


In [5]:
## Project 저장하고, Design 이름 설정 ##

# design 객체가 존재하는 project 객체
proj = M3D.oproject

# # project 저장
# proj.SaveAs(f"{dir}\\{proj_name}.aedt", True)

# design을 원하는 이름으로 변경
M3D.rename_design(desi_name, save=False)

True

In [6]:
## 형상 그리기 ##

# 코일 그리기

origin = [0, 5, 0]
coil = M3D.modeler.create_circle(orientation="XY", origin=origin, radius=0.5, num_sides=12, is_covered=True, name="Coil", material=None, non_model=False)

M3D.modeler.sweep_around_axis(assignment=coil, axis="X", sweep_angle=360, draft_angle=0, number_of_segments=30)

M3D.assign_material(assignment=coil, material="copper")

# 막대 자석 그리기
origin = [-3, -0.5, -0.5]
sizes = [6, 1, 1]
magnet = M3D.modeler.create_box(origin, sizes, name="Magnet", material="NdFe35")


PyAEDT INFO: Modeler class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Materials class has been initialized! Elapsed time: 0m 0sec


c:\Users\abane\anaconda\envs\pyaedt_env\Lib\site-packages\ansys\aedt\core\visualization\advanced\misc.py:43: UserWarning: The PyVista module is required to run functionalities of ansys.aedt.core.visualization.advanced.misc.
Install with 

pip install pyvista
  warnings.warn(


In [7]:
## material 확인 ##

NdFe35 = M3D.materials.exists_material(material="NdFe35")

display(NdFe35.get_magnetic_coercivity())

# # 착자 방향이 다르면 아래 코드로 수정 
# NdFe35.set_magnetic_coercivity(value='-890000A_per_meter', x="1", y="0", z="0")

('-890000A_per_meter', '1', '0', '0')

In [8]:
## coil terminal 설정 ##

# coil terminal sheet 생성

coil_section = []
coil = [coil]   # list에 담아서 for문 사용
for c in coil : 
    M3D.modeler.section(assignment=c, plane="XY", create_new=True, section_cross_object=False)
    coil_section.append( M3D.modeler.sheet_objects[-1] )

M3D.modeler.split(assignment=coil_section, plane="ZX", sides="NegativeOnly", tool=None, split_crossing_objs=False, delete_invalid_objs=True)

# coil terminal에 전류 인가

coil_terminal = []
for s in coil_section :
    coil_terminal.append( M3D.assign_current(assignment=s, amplitude="100A", phase='0deg', solid=False, swap_direction=False, name=None) )


PyAEDT INFO: Boundary Current Current_II6J6C has been created.


In [9]:
## 토크 설정 ##

# 토크 설정

M3D.assign_torque(assignment=magnet, coordinate_system='Global', is_positive=True, is_virtual=True, axis='Z', torque_name="Torque1")


PyAEDT INFO: Boundary Torque Torque1 has been created.


In [10]:
## 자석 회전 ##

 
M3D.modeler.rotate(assignment=coil, axis="Z", angle=45.0, units='deg')
M3D.modeler.rotate(assignment=coil_section, axis="Z", angle=45.0, units='deg')

True

In [11]:
## region 설정 ##

region = M3D.modeler.create_region(pad_value=100, pad_type='Percentage Offset', name='Region')

In [12]:
## Analyze Setup 설정 ##

# setup 객체
my_setup = M3D.create_setup(name="Setup1")

# 현재 solution type에 따른 Analysis setup의 property 확인
# setup 객체의 props 속성에, dictionary 형태로 저장되어있음.
display(my_setup.props)

{'Enabled': True,
 'MeshLink': {'ImportMesh': False},
 'MaximumPasses': 10,
 'MinimumPasses': 2,
 'MinimumConvergedPasses': 1,
 'PercentRefinement': 30,
 'SolveFieldOnly': False,
 'PercentError': 1,
 'SolveMatrixAtLast': True,
 'UseIterativeSolver': False,
 'RelativeResidual': 1e-06,
 'NonLinearResidual': 0.001,
 'SmoothBHCurve': False,
 'MuOption': {'MuNonLinearBH': True},
 'Name': 'Setup1'}

In [13]:
# 위에 display된 props 에서 원하는 부분 수정

my_setup.props['MaximumPasses'] = 10


In [14]:
# analyze 진행
my_setup.analyze()

PyAEDT INFO: Key Desktop/ActiveDSOConfigurations/Maxwell 3D correctly changed.
PyAEDT INFO: Solving design setup Setup1
PyAEDT INFO: Design setup Setup1 solved correctly in 0.0h 0.0m 35.0s


In [15]:
# project 저장
M3D.save_project()

PyAEDT INFO: Project Project1 Saved correctly


True